# Week 4: RES/NRES at 100×100 m (fractions + classification)

Question: does **pooling 10 m AE cells into 100 m tiles** improve agreement with Overture GT?

- Tile = UTM 100 m box; up to **10×10** AE cells.
- GT / preds at 100 m = mean of equal-area 10 m fractions.
- **Classification fair protocol:** balance **tiles** 1:1 (not cell-balanced pooling). See section below + `scripts/build_week4_100m_tiles_balanced.py`.
- Regression / fraction-argmax tracks below are complementary (MAE, imbalanced full tiles).

Rebuild tiles + fraction eval:
```bash
.venv/bin/python -u scripts/build_week4_100m_tiles.py
.venv/bin/python -u scripts/build_week4_100m_tiles_balanced.py
```


In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "outputs" / "week4_100m_fraction_eval" / "summary.json").exists() or (p / "scripts" / "build_week4_100m_tiles.py").exists():
        ROOT = p
        break

DATA = ROOT / "datasets" / "week4_100m_tiles"
OUT = ROOT / "outputs" / "week4_100m_fraction_eval"
manifest = json.loads((DATA / "manifest.json").read_text())
summary = pd.read_csv(OUT / "mae_summary.csv")
f1 = pd.read_csv(OUT / "f1_summary.csv")
per = pd.read_csv(OUT / "mae_per_region_100m.csv")
print("tiles", manifest["n_tiles"], "eligible", manifest["n_tiles_support_ok"], "balanced-eligible", manifest["n_tiles_balanced_ok"])
print("MAE summary")
display(summary.round(4))
print("F1 summary (RES vs NRES)")
display(f1.round(4))


tiles 17151 eligible 4947 balanced-eligible 1281


,mae_frac_res,mae_frac_nres,mae_mean_res_nres,n,mae_frac_empty,scale,model,label,delta_10m_minus_100m,min_cells
0,0.2129,0.2473,0.2301,1281,0.1528,100m,balanced_train,W4 balanced Ridge,0.073,10.0
1,0.3281,0.2781,0.3031,23242,NaN,10m,balanced_train,W4 balanced Ridge,NaN,NaN
2,0.3651,0.1940,0.2796,4947,0.3274,100m,fraction_regression,W3 Ridge,0.063,20.0
3,0.3557,0.3295,0.3426,191422,NaN,10m,fraction_regression,W3 Ridge,NaN,NaN
4,0.3580,0.2091,0.2836,4947,0.3274,100m,pixel_ae,W3 AE LR,0.055,20.0
5,0.3466,0.3305,0.3385,191422,NaN,10m,pixel_ae,W3 AE LR,NaN,NaN
6,0.4433,0.2534,0.3483,4947,0.2984,100m,ghsl,GHSL (soft under tile),0.049,20.0
7,0.4404,0.3542,0.3973,191519,NaN,10m,ghsl,GHSL (soft under tile),NaN,NaN


## Method

```text
tile_i = floor(easting / 100)   # per-metro UTM
tile_j = floor(northing / 100)
frac_res_100 = mean_i frac_res_i     # = total RES area in cells / total cell area
```

Models: W4 balanced Ridge (balanced cells only), W3 Ridge, W3 AE LR, GHSL soft. No retrain — aggregate existing OOF.

Binary label at either scale: `RES` if `frac_res > frac_nres`, else `NRES`; exact ties dropped.

![Cells per tile](../../datasets/week4_100m_tiles/figures/n_cells_per_tile.png)

![Tiles by metro](../../datasets/week4_100m_tiles/figures/tiles_by_region.png)


## Results: 10 m vs 100 m

Pooling helps every model. W3 AE / Ridge land near **0.28** MAE at 100 m on full eligible tiles; GHSL soft ~**0.35**. W4 on the thinner balanced-subset tiles reaches ~**0.23** (not directly comparable to the full-tile ladder).

![10m vs 100m](../../outputs/week4_100m_fraction_eval/figures/mae_10m_vs_100m_ladder.png)

![Delta](../../outputs/week4_100m_fraction_eval/figures/mae_delta_10m_to_100m.png)

![100m ladder](../../outputs/week4_100m_fraction_eval/figures/mae_100m_ladder.png)

![Best AE by metro](../../outputs/week4_100m_fraction_eval/figures/mae_100m_by_metro_best_ae.png)


In [2]:
s100 = summary.loc[summary["scale"] == "100m", ["label", "mae_mean_res_nres", "mae_frac_res", "mae_frac_nres", "n", "delta_10m_minus_100m"]].copy()
s100 = s100.rename(columns={
    "mae_mean_res_nres": "MAE_100m",
    "mae_frac_res": "MAE_res",
    "mae_frac_nres": "MAE_nres",
    "n": "n_tiles",
    "delta_10m_minus_100m": "Δ(10m−100m)",
})
display(s100.round(4).sort_values("MAE_100m"))


,label,MAE_100m,MAE_res,MAE_nres,n_tiles,Δ(10m−100m)
0,W4 balanced Ridge,0.2301,0.2129,0.2473,1281,0.073
2,W3 Ridge,0.2796,0.3651,0.1940,4947,0.063
4,W3 AE LR,0.2836,0.3580,0.2091,4947,0.055
6,GHSL (soft under tile),0.3483,0.4433,0.2534,4947,0.049


## Results: RES vs NRES classification

Label = argmax of RES/NRES fractions (ties dropped). Pooling helps more on the balanced track and for GHSL; full-GT W3 heads are nearly flat.

| Model | F1 10 m | F1 100 m | Δ |
|---|---:|---:|---:|
| W4 balanced Ridge | 0.575 | **0.657** | +0.082 |
| W3 Ridge | 0.522 | 0.526 | +0.004 |
| W3 AE LR | 0.517 | 0.519 | +0.003 |
| GHSL soft | 0.446 | 0.493 | +0.047 |

![F1 10m vs 100m](../../outputs/week4_100m_fraction_eval/figures/f1_10m_vs_100m_ladder.png)

![F1 delta](../../outputs/week4_100m_fraction_eval/figures/f1_delta_10m_to_100m.png)

![F1 100m ladder](../../outputs/week4_100m_fraction_eval/figures/f1_100m_ladder.png)


In [ ]:
b100 = f1.loc[f1["scale"] == "100m", ["label", "macro_f1", "accuracy", "n", "delta_f1_100m_minus_10m"]].copy()
b100 = b100.rename(columns={
    "macro_f1": "F1_100m",
    "n": "n_tiles",
    "delta_f1_100m_minus_10m": "ΔF1(100m−10m)",
})
display(b100.round(4).sort_values("F1_100m", ascending=False))

## Results: LR head on all balanced cells (same as 0.70 chart)

Same ~60k balanced pure cells and binary heads as the Week 4 0.70 ladder. Tile pred = `mean(p_res) ≥ 0.5` (or majority of `y_pred`); GT = majority of cell labels; ties dropped.

| Model | F1 10 m | F1 100 m (min≥1) | Δ | F1 100 m (min≥5) | Δ |
|---|---:|---:|---:|---:|---:|
| W4 balanced LR | **0.704** | 0.668 | −0.036 | **0.763** | +0.059 |
| W3 Ridge | 0.700 | 0.666 | −0.034 | 0.755 | +0.055 |
| W3 AE LR | 0.697 | 0.670 | −0.027 | 0.749 | +0.052 |
| GHSL | 0.692 | 0.588 | −0.104 | 0.749 | +0.058 |

- **min≥1** = every tile that contains any balanced cell (~12.8k tiles) — direct pool of the 0.70 set; slight drop.
- **min≥5** = denser tiles only (~4k) — pooling **beats** cell-level (~0.75–0.76).

![LR min1](../../outputs/week4_100m_fraction_eval/figures/f1_lr_balanced_10m_vs_100m_min1.png)

![LR min5](../../outputs/week4_100m_fraction_eval/figures/f1_lr_balanced_10m_vs_100m_min5.png)


In [ ]:
lr = pd.read_csv(OUT / "f1_lr_balanced_summary.csv")
piv = (
    lr.loc[lr["scale"] == "100m", ["label", "min_cells", "macro_f1", "n", "delta_f1_100m_minus_10m"]]
    .pivot(index="label", columns="min_cells", values=["macro_f1", "delta_f1_100m_minus_10m", "n"])
)
display(lr.loc[lr["scale"] == "100m"].round(4).sort_values(["min_cells", "macro_f1"], ascending=[True, False]))


## Correct protocol: balance 100 m tiles (not cells)

Pooling the **cell-balanced** 10 m set into tiles reuses the wrong sampling unit. Rebuild:

```bash
.venv/bin/python -u scripts/build_week4_100m_tiles_balanced.py
```

1. Label every tile from **full** GT: `y = argmax(frac_res, frac_nres)` (ties dropped).
2. Keep tiles with ≥5 AE cells (primary) or ≥20 (strict).
3. Per metro, undersample to **1:1 RES:NRES tiles** (seed 42).
4. Score with `mean(p_res) ≥ τ` over cells in the tile (W3 AE / W3 Ridge / GHSL have full coverage).

| Model | min≥5 @0.50 | best τ* | min≥20 @0.50 | best τ* |
|---|---:|---:|---:|---:|
| W3 Ridge | **0.707** | 0.715 | 0.646 | 0.656 |
| GHSL | 0.703 | **0.768** | **0.693** | **0.718** |
| W3 AE LR | 0.696 | 0.711 | 0.648 | 0.651 |

![Before](../../datasets/week4_100m_tiles_balanced/figures/imbalanced_tiles_by_region_min5.png)

![After](../../datasets/week4_100m_tiles_balanced/figures/balanced_tiles_by_region_min5.png)

![Ladder](../../outputs/week4_100m_tiles_balanced/figures/f1_balanced_tiles_ladder_min5.png)

![τ curves](../../outputs/week4_100m_tiles_balanced/figures/f1_balanced_tiles_threshold_curves_min5.png)


In [ ]:
from pathlib import Path
BAL_OUT = ROOT / "outputs" / "week4_100m_tiles_balanced"
BAL_DATA = ROOT / "datasets" / "week4_100m_tiles_balanced"
best = pd.read_csv(BAL_OUT / "f1_best_summary.csv")
display(best.loc[best["primary"]].round(4).sort_values(["min_cells", "macro_f1"], ascending=[True, False]))
print("manifest:", json.loads((BAL_DATA / "manifest.json").read_text())["variants"])


## Fair paired: 10 m vs 100 m on the same tiles

Fix the 1:1 balanced tile set. Score cells inside those tiles at 10 m, then the tile aggregate at 100 m (`mean(p) ≥ 0.50`).

| Model | min≥5 · 10 m | 100 m | Δ |
|---|---:|---:|---:|
| W3 Ridge | 0.669 | 0.706 | +0.037 |
| W3 AE LR | 0.664 | 0.696 | +0.032 |
| GHSL | 0.648 | 0.707 | +0.060 |

![Paired](../../outputs/week4_100m_tiles_balanced/figures/f1_paired_10m_vs_100m_min5.png)

![Delta](../../outputs/week4_100m_tiles_balanced/figures/f1_paired_delta_min5.png)


In [ ]:
pair = pd.read_csv(ROOT / "outputs" / "week4_100m_tiles_balanced" / "f1_paired_10m_vs_100m.csv")
display(pair.round(4))


## Takeaways

1. Fair 10 m vs 100 m comparison uses the same 1:1 tile set (section above).
2. At min≥5 / τ=0.50, pooling helps all heads (~+0.03 to +0.06 F1).
3. Fraction-argmax MAE on unbalanced full tiles is a separate track (~0.05 to 0.07 MAE drop).
4. Do not compare the global 10 m cell-balanced 0.70 to an unmatched 100 m number.
